In [1]:
import h3
import pandas as pd
import folium
import branca.colormap as cm
import geopandas as gpd


from shapely.geometry import Polygon

In [2]:
import h3

functions = [f for f in dir(h3) if not f.startswith('_') and callable(getattr(h3, f))]
print(functions)

['H3BaseException', 'H3CellInvalidError', 'H3DirEdgeInvalidError', 'H3DomainError', 'H3DuplicateInputError', 'H3FailedError', 'H3GridNavigationError', 'H3LatLngDomainError', 'H3MemoryAllocError', 'H3MemoryBoundsError', 'H3MemoryError', 'H3NotNeighborsError', 'H3OptionInvalidError', 'H3PentagonError', 'H3ResDomainError', 'H3ResMismatchError', 'H3Shape', 'H3UndirEdgeInvalidError', 'H3ValueError', 'H3VertexInvalidError', 'LatLngMultiPoly', 'LatLngPoly', 'Literal', 'UnknownH3ErrorCode', 'are_neighbor_cells', 'average_hexagon_area', 'average_hexagon_edge_length', 'cell_area', 'cell_to_boundary', 'cell_to_center_child', 'cell_to_child_pos', 'cell_to_children', 'cell_to_children_size', 'cell_to_latlng', 'cell_to_local_ij', 'cell_to_parent', 'cell_to_vertex', 'cell_to_vertexes', 'cells_to_directed_edge', 'cells_to_geo', 'cells_to_h3shape', 'child_pos_to_cell', 'compact_cells', 'directed_edge_to_boundary', 'directed_edge_to_cells', 'edge_length', 'geo_to_cells', 'geo_to_h3shape', 'get_base_cell

In [3]:
lat = 39.73632141579045
lng = -104.9897890191563
resolution = 10

In [4]:
h3_index = h3.latlng_to_cell(lat, lng, resolution)
print(f"H3 Index: {h3_index}")

H3 Index: 8a268cda80dffff


In [5]:



boundary = h3.cell_to_boundary(h3_index)
print(f"Boundary: {boundary}")

Boundary: ((39.73524428010972, -104.98900125244737), (39.73564964864881, -104.98824237067122), (39.73634565833955, -104.98834457890632), (39.7366362996269, -104.98920568004662), (39.736230928869034, -104.9899645663659), (39.73553491904265, -104.98986234700183))


In [8]:
# Create a map
m = folium.Map(location=[lat, lng], zoom_start=16)

# Add the H3 cell to the map
folium.Polygon(locations=boundary, color='blue', weight=5).add_to(m)

# Save the map
m

In [6]:
shapefile_path = r"C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\admin.shp"
gdf = gpd.read_file(shapefile_path)
gdf_wales = gdf[gdf["ctry22nm"] == "Wales"]

# Calculate center of the shapefile for map initialization
center = gdf.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=7)

# Add the shapefile geometry to the map
folium.GeoJson(gdf_wales).add_to(m)

m  # Will render inline in Jupyter

C:\Users\BenasPekarskis\AppData\Local\Temp\ipykernel_46780\4077055951.py:6: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = gdf.geometry.unary_union.centroid


In [7]:
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon
import h3
from h3 import geo_to_h3shape, h3shape_to_cells

def polygon_to_h3_indexes(gdf, resolution):
    """
    Accepts GeoDataFrame with a single polygon or multipolygon.
    Returns a set of H3 indexes at the specified resolution.
    """
    # Union all geometries into one polygon/multipolygon
    geom = gdf.geometry.union_all()

    # Convert to H3 shape and then to cells
    h3_shape = geo_to_h3shape(geom)
    h3_indexes = h3shape_to_cells(h3_shape, resolution)

    return h3_indexes

In [8]:
# Load and filter your shapefile (e.g., Wales)
shapefile_path = r"C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\local_authority_districts_gb.shp"
gdf = gpd.read_file(shapefile_path)
gdf_wales = gdf[gdf["LAD23NM"] == "Aberdeen City"]

# Generate H3 indexes (fast)
resolution = 7
h3_indexes = polygon_to_h3_indexes(gdf_wales, resolution)

# Check the result quickly
print(f"Total H3 indexes generated: {len(h3_indexes)}")
print(list(h3_indexes)[:10])  # print first 5 indexes


Total H3 indexes generated: 42
['8719768deffffff', '8719768ceffffff', '871976b96ffffff', '8719768caffffff', '8719768c6ffffff', '8719768c2ffffff', '871976164ffffff', '8719768ddffffff', '871976160ffffff', '8719768d9ffffff']


In [15]:
import geopandas as gpd
from h3 import geo_to_h3shape, h3shape_to_cells, compact_cells

shapefile_path = r"C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\local_authority_districts_gb.shp"
print(f"Reading shapefile: {shapefile_path}")
resolution = 8

def shapefile_to_compact_h3(shapefile_path, resolution, compact=True):
    """
    Reads a shapefile and returns a set of H3 indexes at the specified resolution.
    Optionally compacts the H3 index set to reduce redundancy.
    """
    # Load the shapefile
    gdf = gpd.read_file(shapefile_path)

    # Filter for a specific district (e.g., Aberdeen City)
    gdf_filterd = gdf[gdf["LAD23NM"] == "Basildon"]

    if gdf_filterd.empty:
        print("No data found for 'Aberdeen City'. Please check the shapefile for the correct name.")
        return []

    # Convert geometry to GeoJSON-like format (needed for geo_to_h3shape)
    # This converts the first geometry into a GeoJSON-like dictionary
    geom = gdf_filterd.geometry.iloc[0].__geo_interface__

    # Convert geometry to H3 shape
    h3_shape = geo_to_h3shape(geom)

    # Rasterize geometry into H3 cells at the given resolution
    h3_indexes = h3shape_to_cells(h3_shape, resolution)

    # Compact the cells if needed
    if compact:
        h3_indexes = compact_cells(h3_indexes)

    return h3_indexes

# Call the function and store the result
h3_indexes = shapefile_to_compact_h3(shapefile_path, resolution, compact=False)

# Now you can safely print the count
print(f"Total H3 indexes generated: {len(h3_indexes)}")
# Your H3 indexes


Reading shapefile: C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\local_authority_districts_gb.shp
Total H3 indexes generated: 167


In [16]:

h3_indexes = h3_indexes

# Convert H3 cells to Shapely polygons
polygons = []
for h in h3_indexes:
    boundary = h3.cell_to_boundary(h)  # list of (lat, lng)
    poly = Polygon([(lng, lat) for lat, lng in boundary])  # convert to (x, y)
    polygons.append(poly)

# Create GeoDataFrame
gdf = gpd.GeoDataFrame(geometry=polygons, crs="EPSG:4326")

# Get map center from first H3 cell
center_lat, center_lng = h3.cell_to_latlng(h3_indexes[0])
m = folium.Map(location=[center_lat, center_lng], zoom_start=10, tiles="cartodbpositron")

# Add polygons to the map
folium.GeoJson(
    gdf,
    style_function=lambda feature: {
        "color": "blue",
        "weight": 1,
        "fillOpacity": 0.3
    }
).add_to(m)

m  # Show map inline in Jupyter


Reading shapefile: C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\aonb.shp


In [47]:
import geopandas as gpd
import json
from h3 import geo_to_h3shape, h3shape_to_cells, compact_cells

# Path to the input shapefile
shapefile_path = r"C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\local_authority_districts_gb.shp"
print(f"Reading shapefile: {shapefile_path}")
resolution = 7

def geometry_to_h3(geometry, resolution, compact=False):
    """
    Converts a single geometry to a set of H3 indexes at the specified resolution.
    Optionally compacts the H3 index set to reduce redundancy.
    """
    try:
        # Convert geometry to GeoJSON-like format
        geom = geometry.__geo_interface__

        # Convert geometry to H3 shape
        h3_shape = geo_to_h3shape(geom)

        # Rasterize geometry into H3 cells at the given resolution
        h3_indexes = h3shape_to_cells(h3_shape, resolution)

        # Compact the cells if needed
        if compact:
            h3_indexes = compact_cells(h3_indexes)

        return h3_indexes
    except Exception as e:
        print(f"Error processing geometry: {e}")
        return []

def shapefile_to_h3(gdf, resolution, compact=False):
    """
    Processes all geometries in a GeoDataFrame and adds H3 indexes to a new 'h3' column as JSON strings.
    """
    # Initialize an empty column for H3 indexes
    gdf['h3'] = None

    # Iterate over each geometry and compute H3 indexes
    for idx, row in gdf.iterrows():
        h3_indexes = geometry_to_h3(row.geometry, resolution, compact)
        # Store as JSON string to handle large lists
        gdf.at[idx, 'h3'] = json.dumps(list(h3_indexes))
        print(f"Processed geometry {idx + 1}/{len(gdf)}: {len(h3_indexes)} H3 indexes")

    return gdf

# Load the shapefile
gdf = gpd.read_file(shapefile_path)

# Handle potential lowercase fid column conflict
if 'fid' in gdf.columns:
    gdf = gdf.rename(columns={'fid': 'feature_id_lower'})

# Reset index to avoid index-related issues
gdf = gdf.reset_index(drop=True)

# Generate H3 indexes for all geometries and add to 'h3' column
gdf_with_h3 = shapefile_to_h3(gdf, resolution, compact=False)

# Print summary
print(f"Total geometries processed: {len(gdf_with_h3)}")
print(f"Columns in output GeoDataFrame: {gdf_with_h3.columns}")

# Save the updated GeoDataFrame to a GeoPackage using pyogrio
output_gpkg = shapefile_path.replace(".shp", "_with_h3_res_7.gpkg")
try:
    # Explicitly map feature_id to FID to avoid conflicts
    gdf_with_h3.to_file(output_gpkg, driver="GPKG", layer_options={'FID': 'feature_id'})
    print(f"Saved output to: {output_gpkg}")
except Exception as e:
    print(f"Error saving GeoPackage: {e}")
    # Fallback: Save GeoDataFrame without h3 column to GeoPackage
    try:
        print("Falling back to saving GeoDataFrame without h3 column...")
        gdf_no_h3 = gdf_with_h3.drop(columns=['h3'])
        gdf_no_h3.to_file(output_gpkg, driver="GPKG", layer_options={'FID': 'feature_id'})
        print(f"Saved GeoDataFrame without h3 column to: {output_gpkg}")
        # Save H3 indexes to a separate JSON file
        # output_json = shapefile_path.replace(".shp", "_h3_indexes.json")
        # h3_dict = {row['feature_id']: json.loads(row['h3']) for _, row in gdf_with_h3.iterrows()}
        # with open(output_json, 'w') as f:
        #     json.dump(h3_dict, f)
        # print(f"Saved H3 indexes to: {output_json}")
    except Exception as e:
        print(f"Error saving fallback GeoPackage: {e}")

Reading shapefile: C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\local_authority_districts_gb.shp
Processed geometry 1/361: 21 H3 indexes
Processed geometry 2/361: 12 H3 indexes
Processed geometry 3/361: 54 H3 indexes
Processed geometry 4/361: 45 H3 indexes
Processed geometry 5/361: 43 H3 indexes
Processed geometry 6/361: 16 H3 indexes
Processed geometry 7/361: 40 H3 indexes
Processed geometry 8/361: 30 H3 indexes
Processed geometry 9/361: 8 H3 indexes
Processed geometry 10/361: 16 H3 indexes
Processed geometry 11/361: 541 H3 indexes
Processed geometry 12/361: 43 H3 indexes
Processed geometry 13/361: 189 H3 indexes
Processed geometry 14/361: 60 H3 indexes
Processed geometry 15/361: 18 H3 indexes
Processed geometry 16/361: 15 H3 indexes
Processed geometry 17/361: 85 H3 indexes
Processed geometry 18/361: 19 H3 indexes
Processed geometry 19/361: 457 H3 indexes
Processed geometry 20/361: 64 H3 indexes
Processed geometry 21/361: 21 H3 indexes
Processed geometry 22/361: 72

In [48]:
import geopandas as gpd
import folium
import json
import h3
from shapely.geometry import Polygon

# Path to the input GeoPackage
gpkg_path = r"C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\local_authority_districts_gb_with_h3_res_7.gpkg"
print(f"Reading GeoPackage: {gpkg_path}")

# Define your filters here (modify as needed)
filters = {
    'LAD23NM': 'Windsor and Maidenhead',  # Example: Filter for Basildon district
    # Add more filters as needed, e.g.:
    # 'Classified': 'some_value',
    # 'LAD23CD': 'some_code'
}

def apply_filters(gdf, filters):
    """
    Applies filters to the GeoDataFrame based on column-value pairs.
    """
    filtered_gdf = gdf.copy()
    for column, value in filters.items():
        if column in filtered_gdf.columns:
            filtered_gdf = filtered_gdf[filtered_gdf[column] == value]
        else:
            print(f"Warning: Column '{column}' not found in GeoDataFrame")
    return filtered_gdf

def h3_to_polygon(h3_index):
    """
    Converts an H3 index to a Shapely Polygon using cell_to_boundary.
    """
    try:
        # Get the boundary of the H3 cell (list of [lat, lng] pairs)
        boundary = h3.cell_to_boundary(h3_index)
        # Create a Shapely Polygon (Folium expects lng, lat order)
        return Polygon([(point[1], point[0]) for point in boundary])
    except Exception as e:
        print(f"Error converting H3 index {h3_index}: {e}")
        return None

def plot_h3_and_geometries(gdf, filters):
    """
    Creates a Folium map with H3 hexagons and filtered geometries for display in a notebook.
    """
    # Apply filters
    filtered_gdf = apply_filters(gdf, filters)
    
    if filtered_gdf.empty:
        print("No data matches the filters. Please check filter values.")
        return None

    # Extract H3 indexes from the h3 column
    all_h3_indexes = []
    for h3_json in filtered_gdf['h3']:
        try:
            h3_indexes = json.loads(h3_json)
            all_h3_indexes.extend(h3_indexes)
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON in h3 column: {e}")
            continue

    print(f"Total H3 indexes to plot: {len(all_h3_indexes)}")

    # Create H3 polygons
    h3_polygons = [h3_to_polygon(h3_idx) for h3_idx in all_h3_indexes]
    h3_polygons = [poly for poly in h3_polygons if poly is not None]  # Filter out None

    if not h3_polygons:
        print("No valid H3 polygons to plot.")
        return None

    # Create a GeoDataFrame for H3 polygons
    h3_gdf = gpd.GeoDataFrame(geometry=h3_polygons, crs=filtered_gdf.crs)

    # Calculate the map center (use centroid of filtered geometries)
    centroid = filtered_gdf.geometry.unary_union.centroid
    map_center = [centroid.y, centroid.x]  # Folium expects [lat, lng]

    # Initialize Folium map
    m = folium.Map(location=map_center, zoom_start=10, tiles="cartodbpositron")

    # Add H3 polygons to the map
    folium.GeoJson(
        h3_gdf.__geo_interface__,
        style_function=lambda x: {
            'fillColor': 'blue',
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.3
        },
        tooltip="H3 Hexagon"
    ).add_to(m)

    # Add filtered geometries to the map
    # folium.GeoJson(
    #     filtered_gdf.__geo_interface__,
    #     style_function=lambda x: {
    #         'fillColor': 'red',
    #         'color': 'black',
    #         'weight': 2,
    #         'fillOpacity': 0.1
    #     },
    #     tooltip=folium.GeoJsonTooltip(fields=['LAD23NM', 'LAD23CD'], aliases=['District Name', 'District Code'])
    # ).add_to(m)

    return m

# Read the GeoPackage
try:
    gdf = gpd.read_file(gpkg_path)
    print(f"Columns in GeoDataFrame: {gdf.columns}")
    print(f"Number of records: {len(gdf)}")
except Exception as e:
    print(f"Error reading GeoPackage: {e}")
    exit()

# Create and display the Folium map
folium_map = plot_h3_and_geometries(gdf, filters)
folium_map  # Display the map in the Jupyter notebook

Reading GeoPackage: C:\Users\BenasPekarskis\Documents\data_mamangement\temp\shape_h3\local_authority_districts_gb_with_h3_res_7.gpkg
Columns in GeoDataFrame: Index(['FID', 'LAD23CD', 'LAD23NM', 'LAD23NMW', 'Classified', 'h3',
       'geometry'],
      dtype='object')
Number of records: 361
Total H3 indexes to plot: 44


C:\Users\BenasPekarskis\AppData\Local\Temp\ipykernel_46780\945701239.py:79: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  centroid = filtered_gdf.geometry.unary_union.centroid
